In [2]:
import os
from IPython.display import display, update_display, Markdown
from openai import OpenAI
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
from dotenv import load_dotenv
load_dotenv(override=True)

import gradio as gr

c:\projects\AI-career-planner\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from jobsearchapi import search_job

job_title = 'AI engineer'
country = 'Singapore'
query = job_title + 'in ' + country

jobs = search_job(query)

print(jobs)

[{'title': 'AI Engineer (FSI, LLM)', 'company': 'Robert Walters', 'description': None}, {'title': 'Applied AI Engineer - LLM & NLP', 'company': 'TOKU LTD.', 'description': None}, {'title': 'Applied AI Engineer, Senior/Staff Fullstack Software Engineer - Singapore', 'company': 'Mistral Ai', 'description': None}, {'title': 'Senior / Software Engineer (AI Computer Vision)', 'company': 'Singapore Technologies Engineering Ltd', 'description': None}, {'title': 'AI Engineer (LLM / Agent Systems)', 'company': 'REVUP CONSULTING PTE. LTD.', 'description': None}, {'title': 'AI Engineer (GenAI / LLM / AI Agents) – Singapore', 'company': 'RevUp Consulting', 'description': None}, {'title': 'Development Engineer AI for Robotic Systems, ARTC', 'company': 'Agency for Science, Technology and Research (A*STAR)', 'description': None}, {'title': 'AI/Machine Learning Engineer (3 years contract)', 'company': 'HAYS SPECIALIST RECRUITMENT PTE. LTD.', 'description': None}, {'title': 'Lead Engineer/ Engineer, Ac

In [4]:
#Extract important features only
search_result = []

for each in jobs:
    if each['description']:
        search_result.append(each['title'] + 'in ' + each['company'])
        search_result.append(each['description'])

print(f'Total job found: {len(jobs) // 2}')

active_job = '\n'.join(search_result)

Total job found: 22


In [5]:
GEMINI_BASE_URL = 'https://generativelanguage.googleapis.com/v1beta/openai/'
GEMINI_API_KEY = os.getenv('GOOGLE_API_KEY')
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=GEMINI_API_KEY)
GEMINI_MODEL = 'gemini-3.1-flash-lite-preview'

In [6]:
OLLAMA_BASE_URL = 'http://localhost:11434/v1'
OLLAMA_API_KEY = 'ollama' # can be any words, not important
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key=OLLAMA_API_KEY)
OLLAMA_MODEL = 'gpt-oss:latest'

In [ ]:
def create_career_plan(job_title, model, stream):
    if not job_title:
        yield '### Please input a valid job title...'
        return

    #Construct system prompt
    system_prompt = """
    You are a helpful education coach to help to design a professional 4 weeks plan for the user based on the job title. 
    If the job title is not valid, please say so and ask user to input the correct job title again. 
    If the job title is valid, Response in markdown. Do not wrap the markdown with code block - reply only in markdown.
    """

    #Construct user prompt
    user_prompt = f"""
    You will be provided a list of extracted similar job from the list below.
    List of active job available in the user country:
    {active_job}
    You will analyze the job title provided by user, think thoroughly before providing a professional 4 weeks plan regarding the structures of the learning path, provide useful link for the learning & tips to achieve success for the user based on the job requirements from similar job provided above.
    Lastly, highlight the most frequent or top few job requirements / skills needed from most company extracted above.
    This is the job title user wish to expertise on:
    Job Title: {job_title}
    """

    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt}
    ]
    selected_model = None

    if model == "Gemini":
        model = GEMINI_MODEL
        selected_model = gemini

    elif model == "Ollama":
        model=OLLAMA_MODEL
        selected_model = ollama
    
    else:
        yield f'### Please select a model before proceeding...'
        return

    print(f'Creating a 4-week plan for {job_title} using model {model}...')
        
    if not stream:
        response = selected_model.chat.completions.create(model=model, messages=messages)
        result = response.choices[0].message.content
        yield "### Your personalized study plan:\n" + result
        return

    else:
        stream = selected_model.chat.completions.create(model=model, messages=messages, stream=True)
        response = "### Your personalized study plan:\n"

        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            yield response


In [41]:
message_input = gr.Textbox(label="Job Title", info="Enter your job title to create study plan", lines=3)

model_selector = gr.Radio(
    choices=["Ollama", "Gemini"],
    value="Ollama",
    label="Select Model"
)

stream_toggle = gr.Checkbox(
    label="Streaming",
    value=True
)

message_output = gr.Markdown()

view = gr.Interface(
    fn=create_career_plan,
    title="AI career planner",
    inputs=[message_input, model_selector, stream_toggle],
    outputs=[message_output],
    examples=[
        ["AI Engineer", "Ollama", True],
        ["Financial Advisor", "Gemini", False]
    ],
    flagging_mode="never"
)

view.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.
